# Validate Orders
Validate raw Orders table for further transformations.

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(DISTINCT order_id) AS unique_orders,
  count(DISTINCT product_id) AS unique_products,
  count(DISTINCT unit_price) AS unique_prices,
  count(DISTINCT process_date) AS unique_process_dates,
  count(DISTINCT source_file) AS unique_source_files,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS missing_orders,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customers,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_products,
  SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS missing_quantities,
  SUM(CASE WHEN unit_price IS NULL THEN 1 ELSE 0 END) AS missing_prices,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS missing_dates,
  SUM(CASE WHEN process_date IS NULL THEN 1 ELSE 0 END) AS missing_process_dates,
  SUM(CASE WHEN source_file IS NULL THEN 1 ELSE 0 END) AS missing_source_files
FROM ecommerce_catalog.bronze.orders_raw;


### **Observations**
- 25 duplicate `order_id` values were identified. Verification confirmed that none of these IDs are `NULL`.
- 25 missing values were identified in `customer_id`.
- 12 missing values were identified in `product_id`.
- 12 missing values were identified in `unit_price`.
- Additionally, there are 232 distinct `product_id` values but 322 distinct `unit_price` values. This anomaly will be investigated further in subsequent steps.


In [0]:
%sql
SELECT product_id, unit_price FROM ecommerce_catalog.bronze.orders_raw
WHERE quantity == 1;

In [0]:
%sql
SELECT unit_price FROM ecommerce_catalog.bronze.orders_raw
WHERE product_id == "P0150"

In [0]:
%sql
SELECT
COUNT(*) AS total_rows_product,
SUM(CASE WHEN unit_price IS NULL THEN 1 ELSE 0 END ) AS null_prices,
SUM(CASE WHEN unit_price < 0 THEN 1 ELSE 0 END) AS negative_prices,
SUM(CASE WHEN unit_price = 12.78 THEN 1 ELSE 0 END) AS normal_price,
SUM(CASE WHEN unit_price <> 12.78 THEN 1 ELSE 0 END) AS abnormal_price
FROM ecommerce_catalog.bronze.orders_raw
WHERE product_id == "P0150";

### **Findings on the `unit_price` Column**
- Following up on the previously identified anomaly (232 distinct products vs. 322 distinct prices), product `P0150` was analyzed as a representative example.
- The analysis of product `P0150` shows that multiple prices exist for the same product.
- Positive price variations could be explained by price updates or discounts, as the source data contains no information regarding discount campaigns.
- In addition, one `NULL` value and two negative prices were found. These represent data quality issues that will be cleaned or handled in the Silver layer.
- In the Bronze layer, these anomalies are only documented; the raw data remains unchanged.

**Column `product_id`:**

The `product_id` column exhibits inconsistent casing and formatting (e.g., uppercase/lowercase variations or leading whitespace).


In [0]:
%sql
SELECT order_date FROM ecommerce_catalog.bronze.orders_raw;

### **Observation:**

The `order_date` column is stored as a string. Multiple different date formats were identified (e.g., `YYYY-MM-DD` and `MM/DD/YYYY`). This inconsistent representation can lead to issues during subsequent transformations or analyses.


### **Action:**

Date format standardization will be performed in the Silver layer, as the raw data is retained unchanged in the Bronze layer.

